# CLOY Processing

This notebook is for you to work with the CLOY data folder.

You will write Python code that works with the content of the file cloy_chronological.jsonl, 

There are two main steps:

1. Turn every CLOY thread (a nested structure of the main post and subsequent comments) into a vector of words (as shown in slides 23-24 of Day 11 notes). [Spend some time looking at the structure of a single thread, since it is nested!]
2. Calculate the similarity score between every two threads  (with cosine similarity) [See sample code below], and save the dataframe into a CSV file.
3. Write a function that given one thread ID, prints out the threadID and subject title of the thread for the top 5 most similar threads.
4. Verification: Use the thread_explorer app to verify that these threads are indeed similar (you can search for the threads and read their content)
5. Do you think stop words are affecting your results?  What happens if you get rid of them? [TextBlob has a list of stop words.]

**Table of Contents**  
1. [CLOY thread to vector of words](#sec1)
2. [Calculating similarity score](#sec2)
3. [Wrapping it into a function](#sec3)
4. [Verification](#sec4)
5. [Accounting for stop words](#sec5)

<a id="sec1"></a>
## 1. CLOY thread to vector of words

Before diving straight into the code, it's helpful to just write down my observations of the structure of each thread.

- Each line is a dictionary
- Keys; post_id, timestamp, author, subreddit, title,.... 
- COMMENTS is its own dictionary within the dictionary and the value is a list of subsequent comments (with the same identifier keys).

The goal: create a vocabulary for all unique words. Each row represents one sentence, and each column represents a unique word. The value represents the number of times the word appears in the original sentence.


The approach:
- open the jsonl 
- create vocab_list
- create docs_list
- for each line: extract the title value. save line into docs_list as a string. for each word in the title, if unique, append it to the vocab_list.
- also extract the comments key and its correspondig list (if such a thing exists)
- within the larger for loop, for every comment in the list: extract the comment. save comment into docs_list as a string. append it to the vocab_list if unique. 

In [2]:
example_dct = {"post_id": "fguorr", "timestamp": 1583920517, "author": "bruisedlee123", "subreddit": "CrashLandingOnYou", "title": "Find someone who’ll make you noodles just like Ri Jeong Hyeok-ssi", "selftext": "", "url": "https://i.redd.it/o8vnfixdn0m41.jpg", "score": 60, "permalink": "https://reddit.com/r/CrashLandingOnYou/comments/fguorr/find_someone_wholl_make_you_noodles_just_like_ri/", "comments": [{"comment_id": "fm1mpbz", "parent_id": "t3_fguorr", "author": "shaneekerr", "comment_body": "😘❤️", "score": 3, "created_utc": 1585681210}, {"comment_id": "fp1776m", "parent_id": "t3_fguorr", "author": "VAOruzhie92", "comment_body": "Man, those noodles looked so good that I went out and bought a bunch of noodles and other Korean food stuffs from my local market.", "score": 3, "created_utc": 1588226355}, {"comment_id": "fqkvlsq", "parent_id": "t3_fguorr", "author": "ednamode101", "comment_body": "I just realized that he's always feeding her (the noodle soup, the coffee from scratch, the bbq in her apartment (when Pyo Chi Su calls him out for giving Seri all the meat) and the best she could do for him was order fried chicken. Hahaha!", "score": 1, "created_utc": 1589437581}]}

In [3]:
example_dct['title']

'Find someone who’ll make you noodles just like Ri Jeong Hyeok-ssi'

We will save that as thread_title (as the first part of the vector).

In [4]:
for comment in example_dct['comments']:
    print(comment['comment_body'])

😘❤️
Man, those noodles looked so good that I went out and bought a bunch of noodles and other Korean food stuffs from my local market.
I just realized that he's always feeding her (the noodle soup, the coffee from scratch, the bbq in her apartment (when Pyo Chi Su calls him out for giving Seri all the meat) and the best she could do for him was order fried chicken. Hahaha!


Since we want to represent each THREAD as a vector, we should extend the vector to include the sub_comments.

I want to make sure that the punctuation is also removed. This is code from Gemini:

In [44]:
import re

def process_text(txt):
    punc = re.compile(r'[^\w\s]')
    cleaned = re.sub(punc, "", txt.lower())
    return cleaned

In [6]:
process_text('Crash Landing On You has been created')

'crash landing on you has been created'

**Getting all threads and all texts**

Now, we are already to tackle the entire jsonl file for just getting the threads. By the end of this, we would want:
- a list of thread_ids
- a list of thread_texts (titles + comments, however many there are)

In [7]:
import json
from textblob import TextBlob

In [38]:
threads_ids = []
threads_text = [] # list of sentences
vocab_set = set() # SET of words


with open ('/Users/crystalzhao/Desktop/~WELLESLEY~/FALL_26/CS 315/week4-tasks/data/RedditExplorer/cloy_chronological.jsonl', 'r', encoding = 'utf-8') as f:
    for line in f:
        data = json.loads(line) # to make sure each json line is handled as a vector
        threads_ids.append(data['post_id'])

        parts = [data.get("title", "")]
        parts.extend(
            comment.get("comment_body", "")
            for comment in data.get("comments", [])
        )
        threads_text.append(" ".join(parts))

        """
        Archives :) 
        cleaned_main, words_main = process_text(data['title'])
        docs_list.append(cleaned_main)
        vocab_set.update(words_main)

        comments = data['comments']
        for c in comments:
            comment = c['comment_body']
            cleaned, words = process_text(comment)
            docs_list.append(cleaned)
            vocab_set.update(words)
        """

In [39]:
len(threads_text)

3075

In [13]:
threads_text[:5]

['Crash Landing On You has been created',
 'ok, let’s get this started! What are your thoughts about the show? i\'m so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn\'t work cuz i\'m thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss. I love this show. I stumbled across it on Netflix, gave it a try, and fell in love. I finished it this morning and already miss it. I love Yoon Se-Ri and Jeong Hyeok and nearly all of the other characters. I was so happy with the ending. I’ve added two more kdramas to my Netflix list. I\'m currently obsessed with Hyun Bin. I\'ve never watched Korean dramas, the most asian movie I ever got to watch was the one by Daniel Henney! And this show comes on Netflix and I\'m awed. I\'ve literally added so many K dramas to my list now!\n\nThis is a well rounded drama- irresp

Good! Each item in the list is a thread.

**Processing all texts so they are standardized**

Testing:

In [32]:
t = 'hEllooo '
t.lower().strip()

from textblob import Word
b = Word(t)
b.lemmatize()

'hEllooo '

Unfortunately, calling textblob methods on every word is simply too computationally expensive, so I am making the choice to not engage with that. 

In [ ]:
processed_threads = []
vector_threads = []
vocab_set = set() # we keep a dictionary of processed unique vocab words 

for raw_t in threads_text:
    vec = {}
    processed_t = process_text(raw_t)
    processed_threads.append(processed_t)

    b = TextBlob(processed_t)

    for w in b.words:
        word = str(w)
        vec[word] = b.word_counts[word]
        if word.isalpha(): # not saving the numbers and other non-character tokens 
            vocab_set.add(word) # just automatically add without any redundancy 
    vector_threads.append(vec)
    """
    b = TextBlob(t)
    for w in b.words:
        w = Word(w).lemmatize()
        vocab_set.update(w)
    """


In [16]:
len(vector_threads)

3075

In [17]:
vector_threads[0]

{'crash': 1,
 'landing': 1,
 'on': 1,
 'you': 1,
 'has': 1,
 'been': 1,
 'created': 1}

In [18]:
vocab_list = sorted(list(vocab_set))
len(vocab_list)

22657

In [20]:
vocab_list[:5]

['a',
 'aa',
 'aaaa',
 'aaaaaaa',
 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaahhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh']

I'm noticing that there are many different spellings / variations of the same word, and ideally, we would use TextBlob to do spelling checks + perform lemmatization, but my computer does not have enough ram for this..  

Approach to making the dataframe:
Iterate through the docs_list and for each comment, iterate through the whole vocab_list to get a dictionary with key (vocab) and value (0 or 1). 

In [21]:
import pandas as pd
from textblob import TextBlob

vector_threads = []

for text in threads_text:
    word_vector = {}
    for token in TextBlob(text).words:
        word = str(token).lower()
        if word.isalpha():
            word_vector[word] = 1
    vector_threads.append(word_vector)

# Rows are threads; columns are words. Missing words have value 0.
df = (
    pd.DataFrame.from_records(vector_threads, index=threads_ids)
    .fillna(0)
    .astype("uint8")
)

In [22]:
df.head()

,crash,landing,on,you,has,been,created,ok,let,s,...,sua,rabbia,totale,sinceramente,ora,scumbag,eundong,distribution,episdode,charecters
dkxewd,1,1,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ffh52z,0,0,1,1,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
fguorr,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
fhcrpl,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
fj5byj,0,0,1,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [23]:
df.shape

(3075, 19242)

<a id="sec2"></a>
## 2. Calculating similarity scores

Using the code from class:

In [24]:
import numpy as np
from numpy.linalg import norm

def cosineSimilarity(vec1, vec2): 
    """Calculate the cosine similarity between the two vectors."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    cosine = np.dot(v1,v2)/(norm(v1)*norm(v2))

    return cosine

Testing the first two vectors:

In [25]:
v1 = df.iloc[0]
v2 = df.iloc[1]

cosineSimilarity(v1, v2)

np.float64(0.03532223996729149)

Now, we want to try using the apply method to calculate the cosineSimilarity between the first row and every other row. axis=1 because we are applying this on all the rows instead of columns.

In [101]:
df.apply(lambda x: cosineSimilarity(x, df.iloc[0]), axis=1)

dkxewd     1.000000
ffh52z     0.035322
fguorr     0.046524
fhcrpl     0.000000
fj5byj     0.076797
             ...   
1s637j8    0.044777
1s7yg7f    0.073078
1s8aeiv    0.034648
1s8i9nn    0.124274
1s8ju38    0.101015
Length: 3075, dtype: float64

Ideally, we will make a square matrix indexed both column and row by the thread_ids.

In [26]:
df.index[0]

'dkxewd'

In [27]:
temp_df = pd.DataFrame()

temp_df[df.index[0]] = df.apply(lambda x: cosineSimilarity(x, df.iloc[0]), axis=1)

In [28]:
temp_df

,dkxewd
dkxewd,1.000000
ffh52z,0.035322
fguorr,0.046524
fhcrpl,0.000000
fj5byj,0.076797
...,...
1s637j8,0.044777
1s7yg7f,0.073078
1s8aeiv,0.034648
1s8i9nn,0.124274


This returns a very helpful series! Now, we just want the final 3075 by 3075 similarity scores table, where the column and row names are both thread_ids, and the values of the table are populated by the cosine similarity scores.

In [29]:
scores = pd.DataFrame()

for i in range(len(processed_threads)):
    index = df.index[i]
    scores[index] = df.apply(lambda x: cosineSimilarity(x, df.iloc[i]), axis=1)

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_32177/1064230607.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[index] = df.apply(lambda x: cosineSimilarity(x, df.iloc[i]), axis=1)
/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_32177/1064230607.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[index] = df.apply(lambda x: cosineSimilarity(x, df.iloc[i]), axis=1)
/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_32177/1064230607.py:5: PerformanceWarning: DataFrame is highly frag

The above code works, but just takes way too long.

In [30]:
scores.head()

,dkxewd,ffh52z,fguorr,fhcrpl,fj5byj,fjimic,fjkurx,fkuhp7,flxz1f,fmbrra,...,1s0qkdq,1s10elb,1s10kr2,1s10qcn,1s13t5y,1s637j8,1s7yg7f,1s8aeiv,1s8i9nn,1s8ju38
dkxewd,1.000000,0.035322,0.046524,0.000000,0.076797,0.042258,0.062348,0.088273,0.071429,0.052926,...,0.056149,0.000000,0.000000,0.000000,0.071112,0.044777,0.073078,0.034648,0.124274,0.101015
ffh52z,0.035322,0.441048,0.201309,0.297368,0.357616,0.224642,0.339150,0.294653,0.264917,0.215922,...,0.338402,0.041794,0.076305,0.174343,0.263742,0.370894,0.336536,0.282708,0.176683,0.049953
fguorr,0.046524,0.201309,1.000000,0.250237,0.216757,0.247717,0.203048,0.239566,0.220990,0.206835,...,0.187432,0.055048,0.100504,0.119808,0.243169,0.211448,0.220145,0.214392,0.161889,0.032898
fhcrpl,0.000000,0.297368,0.250237,1.000000,0.299321,0.237171,0.306186,0.282122,0.267261,0.309421,...,0.265896,0.039528,0.072169,0.186400,0.274391,0.345554,0.299069,0.275487,0.247026,0.070868
fj5byj,0.076797,0.357616,0.216757,0.299321,1.000000,0.242313,0.284894,0.347996,0.319987,0.227613,...,0.306876,0.030289,0.027650,0.197766,0.248483,0.369094,0.320831,0.242138,0.211556,0.090506


In [31]:
scores.to_csv('scores.csv')

I experimented with the code below and it took even longer, so I commented it out.

In [ ]:
"""
scores_list = []

for i in range(len(processed_threads)):
    index = df.index[i]
    scores_dct = df.apply(lambda x: cosineSimilarity(x, df.iloc[i]), axis=1).to_dict()
    dct = {index: scores_dct}
    scores_list.append(dct)

"""
    

Turns out, I must rely on the df because I constructed it that way.

## 3. Wrapping it into a function

In [17]:
import pandas as pd

In [18]:
scores = pd.read_csv('data/scores.csv')

In [19]:
scores.head()

,Unnamed: 0,dkxewd,ffh52z,fguorr,fhcrpl,fj5byj,fjimic,fjkurx,fkuhp7,flxz1f,...,1s0qkdq,1s10elb,1s10kr2,1s10qcn,1s13t5y,1s637j8,1s7yg7f,1s8aeiv,1s8i9nn,1s8ju38
0,dkxewd,1.000000,0.035322,0.046524,0.000000,0.076797,0.042258,0.062348,0.088273,0.071429,...,0.056149,0.000000,0.000000,0.000000,0.071112,0.044777,0.073078,0.034648,0.124274,0.101015
1,ffh52z,0.035322,0.441048,0.201309,0.297368,0.357616,0.224642,0.339150,0.294653,0.264917,...,0.338402,0.041794,0.076305,0.174343,0.263742,0.370894,0.336536,0.282708,0.176683,0.049953
2,fguorr,0.046524,0.201309,1.000000,0.250237,0.216757,0.247717,0.203048,0.239566,0.220990,...,0.187432,0.055048,0.100504,0.119808,0.243169,0.211448,0.220145,0.214392,0.161889,0.032898
3,fhcrpl,0.000000,0.297368,0.250237,1.000000,0.299321,0.237171,0.306186,0.282122,0.267261,...,0.265896,0.039528,0.072169,0.186400,0.274391,0.345554,0.299069,0.275487,0.247026,0.070868
4,fj5byj,0.076797,0.357616,0.216757,0.299321,1.000000,0.242313,0.284894,0.347996,0.319987,...,0.306876,0.030289,0.027650,0.197766,0.248483,0.369094,0.320831,0.242138,0.211556,0.090506


We should set Unnamed:0 as the index.

In [20]:
scores = scores.set_index("Unnamed: 0")
scores = scores.rename_axis('ID')

Let's try to sort the by top similarities with the first ID, and we only need that column --> we woul ideally end up with a series.

In [22]:
temp_top_scores = scores.sort_values(by='dkxewd', ascending=False)['dkxewd']

Since the first row would always be the most similar because they are the same vectors, we want to extract the 2nd to 6th row of this series.

In [23]:
temp_top_scores[1:6]

ID
zoafq7     0.617213
1ntfysd    0.617213
1ntfyai    0.571429
umtgh8     0.571429
fqh4pi     0.566947
Name: dkxewd, dtype: float64

In [24]:
temp_top_scores[1:6].index

Index(['zoafq7', '1ntfysd', '1ntfyai', 'umtgh8', 'fqh4pi'], dtype='object', name='ID')

Voila :D Let's wrap this into a function.

In [25]:
def top_similarities(id):
    results_series = scores.sort_values(by=id, ascending=False)[id].head(6)
    print(results_series)
    return list(results_series[1:6].index)

We can try it on the vector that is most similar to the first one we tried.

In [26]:
top_similarities("zoafq7")

ID
zoafq7     1.000000
1ntfysd    0.666667
dkxewd     0.617213
1ntfyai    0.617213
umtgh8     0.617213
1q56g06    0.577350
Name: zoafq7, dtype: float64


['1ntfysd', 'dkxewd', '1ntfyai', 'umtgh8', '1q56g06']

It's interesting that this second ID is more similar to another vector (in fact, the third most similar ID for the first vector). It's time to validate these.

## 4. Verification

Here's the given ID & its corresponding thread text that I want to look at:    
1rh62zb.   

"comments": [{"comment_id": "o7wchkt", "parent_id": "t3_1rh62zb", "author": "Live-Pen4795", "comment_body": "I loved their imperfect relationship! 💖🫰", "score": 13, "created_utc": 1772294824}, {"comment_id": "o7wlhc9", "parent_id": "t3_1rh62zb", "author": "Fun_Case_8948", "comment_body": "He may be imperfect person but his care for her is so pure and genuine!", "score": 11, "created_utc": 1772297498}, {"comment_id": "o7wolgd", "parent_id": "t3_1rh62zb", "author": "moinonplusjetejure", "comment_body": "Good secondary character development in this series", "score": 16, "created_utc": 1772298430}, {"comment_id": "o7y2p8n", "parent_id": "t3_1rh62zb", "author": "vincentclyde", "comment_body": "Wonderful side plot", "score": 3, "created_utc": 1772313960, "replies": [{"comment_id": "o80fbf7", "parent_id": "t1_o7y2p8n", "author": "CartographerMany4217", "comment_body": "This is what made CLOY the best kdrama I've ever seen. If you took them out it would still be top tier, but making these two start as villains we  cry for by the end is why every part of this series is perfect.", "score": 1, "created_utc": 1772345894}]}, {"comment_id": "o7y3myt", "parent_id": "t3_1rh62zb", "author": "linkypilson", "comment_body": "No title. Just wrong", "score": 1, "created_utc": 1772314257}, {"comment_id": "o7ycpxd", "parent_id": "t3_1rh62zb", "author": "Cookiiesssss", "comment_body": "He was my favorite character and I related to seo dan i cried at the ending for days 🥹", "score": 5, "created_utc": 1772317202, "replies": [{"comment_id": "o80fd68", "parent_id": "t1_o7ycpxd", "author": "CartographerMany4217", "comment_body": "This is what made CLOY the best kdrama I've ever seen. If you took them out it would still be top tier, but making these two start as villains we  cry for by the end is why every part of this series is perfect.", "score": 3, "created_utc": 1772345920}]}, {"comment_id": "o8b1lmo", "parent_id": "t3_1rh62zb", "author": "Portia_Elegant", "comment_body": "ts so funny tho", "score": 1, "created_utc": 1772489116}]}


This thread is mostly a discussion about the relationship between the two characters with many positive sentiments & appreciation expressed. Notice that it is quite long and might have many words in the comments text that dilutes the accuracy of the cosine similarity, but let's try.

In [27]:
import pandas as pd

In [28]:
top_similarities("1rh62zb")

ID
1rh62zb    1.000000
18xdmli    0.366088
fhcrpl     0.330946
kmubp5     0.321984
w6vhtp     0.321596
1h4zsn0    0.321588
Name: 1rh62zb, dtype: float64


['18xdmli', 'fhcrpl', 'kmubp5', 'w6vhtp', '1h4zsn0']

The first thread is "SEO DAN'S MOM", which is very different from the original thread in the sense that this thread explicitly discussees Seo Dan's Mom, but perhaps similar semantically because everyone is expressing how "lovely" she is. 

The second thread is "Captain Ri’s squad! Who’s your favorite?", and I can again see some lexical overlap since the sentiments are overall quite positive and about "favorites", converging with the original thread on the overall one of the comments.

The third thread is "Seung-jun & Dan (CLOY)", which is slightly different in the form of the content since it mentions specific moments from the show and also overall sadder because it is about the "bittersweet ending" of the ship. But upon looking more closely, there are again a lot of similar dramatic phrases that might render the thread similar to the original. It's very evident through this exercise that the order of words does not matter! 

## 5. Stop Words

I think stop words are definitely affecting the cosine similarity results. A definition from Coursera says: *Words like “um,” “like,” and “you know,” that carry little information, are stop words.* Since stop words are so abundant and can be found in any thread - even when it does not contribute to the content in any sense - they can really distort the cosine similarities. Longer documents with more stop words might just be more "similar" to other similar ones, because they share so many stop words.     

Knowing this, let's try to remove stop words from both the vocabulary and thread texts, and rerun everything.

**NOTE**: I'm having trouble finding how to access the stop words with just TextBlob, so I want to try using nltk.

In [34]:
import json
from textblob import TextBlob

Let's write a function to remove stop words. Let's remind ourselves what does process_text look like:

In [51]:
import re

def process_text(txt):
    punc = re.compile(r'[^\w\s]')
    processed = re.sub(punc, "", txt.lower())
    return processed

In [52]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

def clean_text(text):
    processed = process_text(text)
    cleaned = []
    for w in processed.split(' '):
        if w not in stopwords.words('english'): # i had to look up the exact wording
            cleaned.append(w)
    return cleaned

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/crystalzhao/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [53]:
clean_text('Does this function.... even work')

['function', 'even', 'work']

In [54]:
threads_text_cleaned = [] # list of sentences
vocab_set_cleaned = set() # SET of words

# let's just use the threads_text we already have 

for thread in threads_text:
    cleaned_tokens = clean_text(thread)
    threads_text_cleaned.append(" ".join(cleaned_tokens))
    vocab_set_cleaned.update(cleaned_tokens)

It will likely be too computationally expensive to try to calculate all the similarity scores again, so I will leave this at this.